# Multi-Tenant Orchestrator with OpenAI Agents SDK (`MCPServerSse`)

This notebook demonstrates the standard, native way to integrate the **Multi-Tenant MCP Orchestrator** with the **OpenAI Agents SDK** using its built-in `MCPServerSse` transport.

By using `MCPServerSse`, you do not need to manually handle JSON-RPC handshakes or custom event loops. The OpenAI Agents SDK automatically manages full-duplex communications with the dynamically spawned container over the remote HTTP/SSE gateway.

## 1. Setup Environment
We load API keys from our `.env` configuration.

In [1]:
import os
from pathlib import Path
import httpx
from dotenv import load_dotenv

# Load environment variables
load_dotenv(Path("..") / ".env")

GATEWAY_URL = "http://localhost:8000"
API_KEY = "super-secret-gateway-key"

print(f"Gateway Endpoint: {GATEWAY_URL}")

Gateway Endpoint: http://localhost:8000


## 2. Request an Isolated Workspace Session
We make a quick POST request to the Gateway to spawn our secure sandbox container.

In [2]:
headers = {
    "Authorization": f"Bearer {API_KEY}",
    "Content-Type": "application/json"
}

with httpx.Client() as client:
    res = client.post(f"{GATEWAY_URL}/api/sessions", headers=headers, timeout=20.0)
    assert res.status_code == 200, f"Failed to create session: {res.text}"
    session_id = res.json()["session_id"]
    
print(f"\u2705 Isolated sandbox spawned with Session ID: {session_id}")

✅ Isolated sandbox spawned with Session ID: aa2582a779f741ca991188338be41f2f


## 3. Configure Transport, Instantiate Agent, and Execute Mission
Because the `MCPServerSse` transport consumes its connection lifecycle during execution, we declare, connect, and run the agent in a single self-contained flow. This guarantees a fresh connection state and prevents any notebook variable reuse issues.

In [3]:
from agents import Agent, Runner, RunConfig
from agents.mcp import MCPServerSse, MCPServerSseParams
from openai import AsyncOpenAI
from agents import OpenAIChatCompletionsModel

# 1. Configure the remote HTTP/SSE Server Transport (fresh instance per execution)
sse_url = f"{GATEWAY_URL}/mcp/{session_id}/sse"
params = MCPServerSseParams(url=sse_url, headers=headers)
mcp_server = MCPServerSse(params=params, name="Remote Sandboxed Workspace")

# 2. Initialize the OpenAI Agent with remote capabilities
model_name = os.environ.get("DEFAULT_MODEL", "google/gemini-2.0-flash-001")
agent = Agent(
    name="RemoteCoder",
    instructions="You are a remote coding assistant with secure, sandboxed terminal and filesystem capabilities.",
    model=OpenAIChatCompletionsModel(
        model=model_name.replace("openrouter/", ""),
        openai_client=AsyncOpenAI(
            base_url="https://openrouter.ai/api/v1",
            api_key=os.environ.get("OPENROUTER_API_KEY")
        )
    ),
    mcp_servers=[mcp_server]
)

async def execute_mission(mission: str):
    print(f"🚀 Deploying Agent on Mission: {mission}\n")
    
    async with mcp_server:
        stream = Runner.run_streamed(agent, mission, max_turns=10, run_config=RunConfig())
        async for event in stream.stream_events():
            if event.type == "run_item_stream_event":
                item = event.item
                if event.name == "tool_called":
                    print(f"\n🛠️  [TOOL CALL] {item.raw_item.name}({item.raw_item.arguments})")
                elif event.name == "tool_output":
                    print(f"✅ [RESULT] {str(item.output)[:200]}...")
            elif event.type == "raw_response_event":
                from openai.types.responses import ResponseTextDeltaEvent
                if isinstance(event.data, ResponseTextDeltaEvent):
                    print(event.data.delta, end="", flush=True)

# Run a sample coding mission!
await execute_mission(
    "Write a python file hello.py that calculates factorials, "
    "execute it using run_bash, and report the output."
)

🚀 Deploying Agent on Mission: Write a python file hello.py that calculates factorials, execute it using run_bash, and report the output.


🛠️  [TOOL CALL] write_file({"filepath":"hello.py","content":"def factorial(n):\n    if n == 0 or n == 1:\n        return 1\n    else:\n        return n * factorial(n - 1)\n\nif __name__ == \"__main__\":\n    nums = [0, 5, 10]\n    for n in nums:\n        print(f\"The factorial of {n} is {factorial(n)}\")\n"})
✅ [RESULT] {'type': 'text', 'text': 'Successfully wrote 232 bytes to hello.py'}...

🛠️  [TOOL CALL] run_bash({"command":"python3 hello.py"})
✅ [RESULT] {'type': 'text', 'text': '[Exit code: 0]\nThe factorial of 0 is 1\nThe factorial of 5 is 120\nThe factorial of 10 is 3628800\n'}...
I have created the Python file `hello.py` which calculates factorials and executed it. Here is the result:

**File Content (`hello.py`):**
```python
def factorial(n):
    if n == 0 or n == 1:
        return 1
    else:
        return n * factorial(n - 1)

if __name_

## 4. Terminate and Clean up Session
Once the workflow finishes, we request the Orchestrator to delete the session, which cleanly reaps the container and workspace folders.

In [ ]:
with httpx.Client() as client:
    res = client.delete(f"{GATEWAY_URL}/api/sessions/{session_id}", headers=headers, timeout=20.0)
    print(f"\u2705 Terminated dynamic session: {res.json()}")